# RS-Solve: Full Benchmark Generation (Kaggle Cloud Worker)
**Đề tài nghiên cứu:** Dự đoán có chọn lọc dựa trên khả năng phân giải cho MLLM ảnh viễn thám (RGSP - IEEE TGRS).

Notebook này tự động quét các bộ dữ liệu viễn thám chuẩn hóa gắn trong `/kaggle/input/` và sinh tập câu hỏi trắc nghiệm quy mô lớn (**Mục tiêu: 30.000 - 40.000 mẫu**) theo đúng chuẩn thiết kế khoa học của bài báo.

In [ ]:
# 1. Tải mã nguồn mới nhất từ GitHub
import os, sys, shutil, subprocess
from pathlib import Path

repo_dir = Path("/kaggle/working/RGSP-nckh")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

!git clone https://github.com/kttt294/RGSP-nckh.git
!pip install -q Pillow
print("Clone và cài đặt môi trường thành công!")

In [ ]:
# 2. Kiểm tra các bộ dữ liệu đã được mount trong /kaggle/input
input_dir = Path("/kaggle/input")
print("=== DANH SÁCH BỘ DỮ LIỆU ĐÃ GẮN TRONG /kaggle/input ===")
if input_dir.is_dir():
    for sub in sorted(input_dir.iterdir()):
        if sub.is_dir():
            print(f"  [+] {sub.name}")

In [ ]:
# 3. Chạy toàn bộ bộ test kiểm thử chất lượng trước khi sinh dữ liệu
os.chdir("/kaggle/working/RGSP-nckh/rs-solve")
!python -m unittest test_parsers.py
!python -m unittest test_label_parser.py

In [ ]:
# 5. Thống kê và kiểm tra kết quả xuất ra
import json
report_file = Path("/kaggle/working/rs_solve_full_benchmark_build_report.json")
jsonl_file = Path("/kaggle/working/rs_solve_full_benchmark.jsonl")
pool_file = Path("/kaggle/working/rs_solve_full_benchmark_full_pool.jsonl")

if jsonl_file.exists():
    total_q = sum(1 for _ in jsonl_file.open(encoding="utf-8"))
    size_mb = jsonl_file.stat().st_size / (1024 * 1024)
    print("=== KẾT QUẢ BENCHMARK CÂN BẰNG ĐA NGUỒN ===")
    print(f"File benchmark: {jsonl_file.name}")
    print(f"Tổng số câu hỏi chọn lọc: {total_q:,} câu")
    print(f"Dung lượng file: {size_mb:.2f} MB")

if pool_file.exists():
    total_pool = sum(1 for _ in pool_file.open(encoding="utf-8"))
    pool_mb = pool_file.stat().st_size / (1024 * 1024)
    print(f"\nFile Full Pool: {pool_file.name}")
    print(f"Tổng số câu hỏi trong full pool: {total_pool:,} câu")
    print(f"Dung lượng pool: {pool_mb:.2f} MB")

if report_file.exists():
    report = json.loads(report_file.read_text(encoding="utf-8"))
    print("\nPhân bổ số lượng theo từng loại câu hỏi (Q1 - Q6):")
    for k, v in report.get("counts_by_kind", {}).items():
        print(f"  - {k:8s}: {v:,} câu")
    if report.get("selected_by_dataset_and_kind"):
        print("\nPhân bổ câu hỏi theo từng dataset:")
        for k, ds_map in report["selected_by_dataset_and_kind"].items():
            print(f"  {k:8s}: {dict(ds_map)}")
    print(f"\nTổng số scene/ảnh đã xử lý: {report.get('scenes_processed', 0):,}")
    if report.get("dataset_scene_counts"):
        print("Phân bổ scene theo dataset:")
        for ds, c in report["dataset_scene_counts"].items():
            print(f"  - {ds.upper()}: {c:,} scene")
    if report.get("rejects_count"):
        print(f"\nTổng số ứng viên bị lọc bởi QA: {report['rejects_count']:,} lần")


In [ ]:
# 5. Thống kê và kiểm tra kết quả xuất ra
import json
report_file = Path("/kaggle/working/rs_solve_full_benchmark_build_report.json")
jsonl_file = Path("/kaggle/working/rs_solve_full_benchmark.jsonl")

if jsonl_file.exists():
    total_q = sum(1 for _ in jsonl_file.open(encoding="utf-8"))
    size_mb = jsonl_file.stat().st_size / (1024 * 1024)
    print("=== KẾT QUẢ SINH TẬP DỮ LIỆU ===")
    print(f"File benchmark: {jsonl_file.name}")
    print(f"Tổng số câu hỏi sinh được: {total_q:,} câu")
    print(f"Dung lượng file: {size_mb:.2f} MB")

if report_file.exists():
    report = json.loads(report_file.read_text(encoding="utf-8"))
    print("\nPhân bổ số lượng theo từng loại câu hỏi (Q1 - Q6):")
    for k, v in report.get("counts_by_kind", {}).items():
        print(f"  - {k:8s}: {v:,} câu")
    print(f"\nTổng số scene/ảnh đã xử lý: {report.get('scenes_processed', 0):,}")
    if report.get("dataset_scene_counts"):
        print("Phân bổ scene theo dataset:")
        for ds, c in report["dataset_scene_counts"].items():
            print(f"  - {ds.upper()}: {c:,} scene")
    if report.get("rejects_count"):
        print(f"\nTổng số ứng viên bị lọc bởi QA: {report['rejects_count']:,} lần")